Run off-the-shelf Depth Anything V2 Small on the local dataset and compute metrics.

Expects data layout produced by download_data.py:
    data/raw/rgb/     — input RGB images
    data/raw/depth/   — ground-truth depth maps (same stems, .png)

Predictions are saved as .npy arrays under results/predictions/.
A metrics summary is printed and saved to results/metrics.json.

In [1]:
# Imports and constants

import json
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoModelForDepthEstimation, AutoProcessor

sys.path.insert(0, str(Path("..").resolve()))
from src.utils import load_gt_depth, align_scale_shift
from src.eval import abs_rel, rmse, threshold_accuracy

MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
RGB_DIR = Path("../data/raw/test/rgb")
DEPTH_DIR = Path("../data/raw/test/depth")
PRED_DIR = Path("../results/predictions")
METRICS_PATH = Path("../results/metrics.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
#infrence function
def run_inference(
    model: AutoModelForDepthEstimation,
    processor: AutoProcessor,
    image: Image.Image,
) -> np.ndarray:
    print(f"Running inference on image.")
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.predicted_depth.squeeze(0).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    pred_resized = torch.nn.functional.interpolate(
        pred,
        size=(image.height, image.width),
        mode="bilinear",
        align_corners=False,
    )
    print(f"Predicted depth map shape: {pred_resized.shape}")
    return pred_resized.squeeze().cpu().numpy().astype(np.float32)


In [3]:
# Run inference and eval per sample

print(f"main function started")
PRED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Loading model on {DEVICE}…")
model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

rgb_paths = sorted(RGB_DIR.glob("*"))
if not rgb_paths:
    raise FileNotFoundError(f"No images found in {RGB_DIR}. Run download_data.py first.")

all_metrics: list[dict] = []


print(f"Inference started")

for rgb_path in tqdm(rgb_paths, desc="Inference"):
    gt_path = DEPTH_DIR / (rgb_path.stem + ".npy")
    if not gt_path.exists():
        print(f"  [warn] no GT depth for {rgb_path.name}:{gt_path}, skipping")
        continue

    image = Image.open(rgb_path).convert("RGB")
    gt = load_gt_depth(gt_path)

    pred = run_inference(model, processor, image)

    if pred.shape != gt.shape:
        gt_t = torch.from_numpy(gt).unsqueeze(0).unsqueeze(0)
        gt = torch.nn.functional.interpolate(
            gt_t, size=pred.shape, mode="bilinear", align_corners=False
        ).squeeze().numpy()

    pred_aligned = align_scale_shift(pred, gt)
    pred_aligned = np.clip(pred_aligned, 0, None)

    np.save(PRED_DIR / (rgb_path.stem + ".npy"), pred_aligned)

    print(f"Saving predicted depth map to {PRED_DIR / (rgb_path.stem + '.npy')}")

    sample_metrics = {
        "file": rgb_path.name,
        "abs_rel": abs_rel(pred_aligned, gt),
        "rmse": rmse(pred_aligned, gt),
        "delta1": threshold_accuracy(pred_aligned, gt, 1.25),
        "delta2": threshold_accuracy(pred_aligned, gt, 1.25**2),
        "delta3": threshold_accuracy(pred_aligned, gt, 1.25**3),
    }
    all_metrics.append(sample_metrics)

main function started
Loading model on cpu…


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Inference started


Inference:   0%|          | 0/12 [00:00<?, ?it/s]

Running inference on image.


...

Inference:  92%|█████████▏| 11/12 [00:13<00:01,  1.09s/it]

Predicted depth map shape: torch.Size([1, 1, 320, 320])
Aligning scale and shift
Saving predicted depth map to ../results/predictions/009891.npy
Running inference on image.


Inference: 100%|██████████| 12/12 [00:14<00:00,  1.09s/it]

Inference: 100%|██████████| 12/12 [00:14<00:00,  1.22s/it]

Predicted depth map shape: torch.Size([1, 1, 320, 320])
Aligning scale and shift
Saving predicted depth map to ../results/predictions/009980.npy


In [4]:
# Run eval
if not all_metrics:
    print("No matched RGB/depth pairs found.")
else:
    summary = {
        "n_samples": len(all_metrics),
        "abs_rel": float(np.mean([m["abs_rel"] for m in all_metrics])),
        "rmse": float(np.mean([m["rmse"] for m in all_metrics])),
        "delta1": float(np.mean([m["delta1"] for m in all_metrics])),
        "delta2": float(np.mean([m["delta2"] for m in all_metrics])),
        "delta3": float(np.mean([m["delta3"] for m in all_metrics])),
    }

    print("\n--- Baseline metrics (pre-finetuning) ---")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    with open(METRICS_PATH, "w") as f:
        json.dump({"summary": summary, "per_sample": all_metrics}, f, indent=2)

    print(f"\nMetrics saved to {METRICS_PATH}")
    print(f"Predictions saved to {PRED_DIR}")


--- Baseline metrics (pre-finetuning) ---
  n_samples: 12
  abs_rel: 0.8453074395656586
  rmse: 27.281686147054035
  delta1: 0.14637532552083332
  delta2: 0.30886067708333326
  delta3: 0.501083984375

Metrics saved to ../results/metrics.json
Predictions saved to ../results/predictions
